# 00 · เตรียมข้อมูลและตัวแปร (Data preparation)

Notebook นี้ดาวน์โหลดข้อมูลสาธารณะของประเทศไทย แปลงให้อยู่ในรูปที่แบบจำลองใช้ และสร้าง persona ของตัวแทนประชาชน
(แนวทางโครงงานหัวข้อ 5, 9.2 และระยะที่ 1)

| ใช้ทำอะไรในแบบจำลอง | แหล่งข้อมูล | ไฟล์ผลลัพธ์ |
|---|---|---|
| ระดับการสัมผัสตามสถานที่ (บ้าน/งาน/โรงเรียน/อื่น ๆ) → $c_a$ | Prem et al. (2021) synthetic contact matrices, THA | `data/processed/contact_by_location_TH.csv` |
| สัดส่วนอายุ 20–64 ปี สำหรับ persona | UN population 2020 (ไฟล์เดียวกับ Prem) | `data/processed/age_groups_TH_20_64.json` |
| ขอบเขตการลดการเคลื่อนที่นอกบ้าน (ใช้กำหนดช่วง sensitivity) | Google COVID-19 Community Mobility Reports, TH | `data/processed/mobility_TH_national.csv`, `mobility_TH_summary.csv` |
| ขอบเขตระดับนโยบาย + เส้นผู้ป่วยอ้างอิง (บริบท ไม่ใช่ calibration) | OxCGRT national, THA (มี `ConfirmedCases`) | `data/processed/oxcgrt_TH.csv` |
| สัดส่วนอาชีพ / ทำงานจากบ้านได้ / ขนาดครัวเรือน | NSO LFS / Census 2020 (กรอกเองใน `config/persona_marginals.csv`) | `config/personas.csv` |

> ทุกค่าที่ยังไม่มีแหล่งข้อมูลจะถูกบันทึกเป็น **assumption** ใน `data/parameter_sources.csv` ห้ามนำเสนอเป็นข้อมูลจริง

Notebook นี้ไม่ต้องใช้ GPU (รันบน CPU ใน Colab หรือในเครื่องได้)

In [ ]:
# ---- Setup (same cell in every notebook) ----
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ThaiGABM")   # where you uploaded the abm folder
    IN_COLAB = True
except ImportError:                                   # running locally (notebooks 00 and 01)
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False

assert (ROOT / "gabm").exists(), f"gabm/ not found in {ROOT}: upload the abm folder to MyDrive/ThaiGABM"
sys.path.insert(0, str(ROOT))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)

from gabm import load_config
cfg = load_config(ROOT / "config" / "base.yaml")
FIG = ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| Colab:", IN_COLAB)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "no GPU")

In [ ]:
import json, zipfile, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt

RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True); PROC.mkdir(parents=True, exist_ok=True)

def fetch(url, dest):
    """Download once. If the download fails, upload the file manually to data/raw/ and rerun."""
    dest = RAW / dest
    if dest.exists() and dest.stat().st_size > 0:
        print("cached  ", dest.name); return dest
    try:
        print("download", url)
        urllib.request.urlretrieve(url, dest)
    except Exception as e:
        raise RuntimeError(f"Download failed ({e}). Download {url} by hand and put it at {dest}") from e
    return dest

PREM = "https://raw.githubusercontent.com/kieshaprem/synthetic-contact-matrices/master/"
SOURCES = {
    "contact_home.rdata":   PREM + "output/syntheticmatrices/contact_home.rdata",
    "contact_work.rdata":   PREM + "output/syntheticmatrices/contact_work.rdata",
    "contact_school.rdata": PREM + "output/syntheticmatrices/contact_school.rdata",
    "contact_others.rdata": PREM + "output/syntheticmatrices/contact_others.rdata",
    "popage_total2020.csv": PREM + "generate_synthetic_matrices/input/pop/popage_total2020.csv",
    "Region_Mobility_Report_CSVs.zip": "https://www.gstatic.com/covid19/mobility/Region_Mobility_Report_CSVs.zip",
    "OxCGRT_compact_national_v1.csv": "https://raw.githubusercontent.com/OxCGRT/covid-policy-dataset/main/data/OxCGRT_compact_national_v1.csv",
}
files = {name: fetch(url, name) for name, url in SOURCES.items()}

## 1. Contact matrices ของไทย (Prem et al., 2021)

แต่ละไฟล์เป็น list ของ R ที่มีเมทริกซ์ 16×16 ต่อประเทศ (กลุ่มอายุ 0–4, 5–9, …, 75+)
ผลรวมแถว $i$ = จำนวนผู้สัมผัสเฉลี่ยต่อวันของคนอายุกลุ่ม $i$ ในสถานที่นั้น
เราเฉลี่ยแบบถ่วงน้ำหนักประชากรสำหรับผู้ใหญ่ 20–64 ปี (ตัวแทนประชาชนเป็นผู้ใหญ่) และทุกอายุ

In [ ]:
import rdata

pop = pd.read_csv(files["popage_total2020.csv"])
tha = pop[pop.iloc[:, 0].str.strip() == "Thailand"].iloc[0]
ages = [c for c in pop.columns if c.startswith("age")]
p5 = tha[ages].astype(float).values * 1000                        # thousands -> persons, 5-year groups to 100+
pop16 = np.append(p5[:15], p5[15:].sum())                          # Prem uses 16 groups, last = 75+
print(f"Thailand population 2020 (UN): {p5.sum()/1e6:.1f} M")

ADULT = slice(4, 13)                                                # 20-24 ... 60-64
rows = []
for loc in ["home", "work", "school", "others"]:
    M = rdata.read_rda(files[f"contact_{loc}.rdata"])[f"contact_{loc}"]["THA"]
    per_age = np.asarray(M).sum(axis=1)
    rows.append({"location": loc,
                 "all_ages": np.average(per_age, weights=pop16),
                 "adult_20_64": np.average(per_age[ADULT], weights=pop16[ADULT])})
contacts = pd.DataFrame(rows)
contacts.to_csv(PROC / "contact_by_location_TH.csv", index=False)
contacts.round(2)

In [ ]:
# Age distribution of adults 20-64 for persona sampling
a = dict(zip(range(4, 13), pop16[4:13]))
age_groups = {"20-29": a[4] + a[5], "30-39": a[6] + a[7], "40-49": a[8] + a[9], "50-64": a[10] + a[11] + a[12]}
tot = sum(age_groups.values()); age_groups = {k: round(v / tot, 4) for k, v in age_groups.items()}
json.dump(age_groups, open(PROC / "age_groups_TH_20_64.json", "w"))
age_groups

## 2. ระดับการสัมผัสของแต่ละ action

ใช้สูตรใน `gabm/contacts.py` (แนวทางหัวข้อ 5.3) ค่า `f_*` และ `baseline_mix` ใน `config/base.yaml` เป็น **สมมติฐาน**
ที่จะทดสอบช่วงค่าใน notebook 04 ค่าที่ได้จากข้อมูลจริงคือจำนวนผู้สัมผัสตามสถานที่ด้านบน

In [ ]:
from gabm.sim import contact_model_from_cfg
cm = contact_model_from_cfg(cfg, ROOT)
print(f"home contacts/day = {cm.home:.2f}   normal-day out-of-home = {cm.base_out:.2f}   w_home = {cm.w_home:.3f}")
cm.table().round(3)

## 3. Google Community Mobility (ไทย ระดับประเทศ, 2020–2022)

ใช้ดูว่าในช่วงมาตรการจริง การไปที่ทำงาน/ร้านค้าลดลงได้มากแค่ไหน เพื่อตั้ง **ขอบเขต** ของ $q_{out}$ ที่สมเหตุสมผล
ไม่ได้ใช้ fit แบบจำลอง

In [ ]:
z = zipfile.ZipFile(files["Region_Mobility_Report_CSVs.zip"])
mob = pd.concat([pd.read_csv(z.open(n), parse_dates=["date"]) for n in z.namelist() if "_TH_" in n])
mob = mob[mob["sub_region_1"].isna()].sort_values("date")
cols = {c: c.replace("_percent_change_from_baseline", "") for c in mob.columns if c.endswith("_baseline")}
mob = mob[["date", *cols]].rename(columns=cols)
mob.to_csv(PROC / "mobility_TH_national.csv", index=False)

roll = mob.set_index("date").rolling(7).mean()
summary = pd.DataFrame({"min_7d": roll.min(), "p10_7d": roll.quantile(0.10), "median_7d": roll.median(),
                        "date_of_min": roll.idxmin().dt.date})
summary.to_csv(PROC / "mobility_TH_summary.csv")
display(summary.round(1))

ax = roll.plot(figsize=(11, 3.5), lw=1); ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("% change vs Jan-Feb 2020 baseline (7-day mean)"); ax.set_title("Thailand, Google Community Mobility")
plt.tight_layout(); plt.savefig(FIG / "00_mobility_TH.png", dpi=150); plt.show()

In [ ]:
# Rough check of q_out bounds: workplace + retail drop at the strictest week vs the model's own range.
w, r = summary.loc["workplaces", "min_7d"], summary.loc["retail_and_recreation", "min_7d"]
print(f"Strictest week: workplaces {w:.0f}%, retail/recreation {r:.0f}% -> out-of-home activity roughly x{1 + (w + r) / 200:.2f}")
q_all_home = cm.q_out({"stay_home": 1, "work_onsite": 0, "essential_errand": 0, "visit_family": 0})
print(f"Model range: q_out = {q_all_home:.2f} (everyone stays home) .. 1.00 (normal day mix) .. above 1 if more go out")

## 4. OxCGRT ประเทศไทย: ระดับความเข้มงวดของนโยบายและผู้ป่วยยืนยัน

ใช้ (1) ตรวจว่าระดับ `d` ของนโยบายใน config อยู่ในช่วงที่ไทยเคยใช้จริง และ (2) เป็นเส้นผู้ป่วยอ้างอิงเชิงคุณภาพ
**ไม่ใช่** ข้อมูลสำหรับ calibration เพราะประชากร ระยะเวลา และนิยามต่างจากแบบจำลอง

In [ ]:
ox = pd.read_csv(files["OxCGRT_compact_national_v1.csv"], low_memory=False)
ox = ox[(ox["CountryCode"] == "THA") & (ox["Jurisdiction"] == "NAT_TOTAL")].copy()
ox["date"] = pd.to_datetime(ox["Date"].astype(str))
keep = ["date", "StringencyIndex_Average", "ConfirmedCases", "ConfirmedDeaths"] + [c for c in ox.columns if c[:2] in {"C1","C2","C4","C6","C7"} and "Flag" not in c]
ox = ox[keep].sort_values("date")
ox["new_cases"] = ox["ConfirmedCases"].diff().clip(lower=0)
ox["new_cases_per100k_7d"] = ox["new_cases"].rolling(7).mean() * 1e5 / p5.sum()
ox.to_csv(PROC / "oxcgrt_TH.csv", index=False)

print("Stringency index (0-100) quantiles:", ox["StringencyIndex_Average"].quantile([0.1, 0.5, 0.9]).round(1).to_dict())
for k, v in cfg["policy"]["levels"].items():
    print(f"  config level {k:8s} d={v['d']:.1f} -> comparable stringency ~{100 * v['d']:.0f}")

fig, ax1 = plt.subplots(figsize=(11, 3.5))
ax1.plot(ox["date"], ox["StringencyIndex_Average"], color="#4C78A8"); ax1.set_ylabel("OxCGRT stringency", color="#4C78A8")
ax2 = ax1.twinx(); ax2.plot(ox["date"], ox["new_cases_per100k_7d"], color="#F58518"); ax2.set_ylabel("new cases /100k/day (7d)", color="#F58518")
ax1.set_title("Thailand: policy stringency and reported cases (context only)")
plt.tight_layout(); plt.savefig(FIG / "00_oxcgrt_TH.png", dpi=150); plt.show()

## 5. สร้าง persona ตัวแทนประชาชน

สุ่มแบบแบ่งชั้น (stratified) ตามกลุ่มอาชีพใน `config/persona_marginals.csv` ด้วย seed คงที่
น้ำหนัก `weight` ของแต่ละ agent = สัดส่วนกลุ่ม / จำนวน agent ในกลุ่ม ใช้รวมผล action
ตอนนี้สัดส่วนอาชีพยังเป็น **assumption**: แก้คอลัมน์ `share`, `p_can_wfh` และ `status`/`source_note` เมื่อได้ตาราง NSO แล้วรัน cell นี้ใหม่

In [ ]:
from gabm.agents import make_personas, persona_text

marg = pd.read_csv(ROOT / cfg["personas"]["marginals"])
display(marg)
pcfg = dict(cfg["personas"], age_groups=age_groups)          # use UN age structure instead of the config fallback
personas = make_personas(marg, pcfg, cfg["sim"]["n_agents"], cfg["sim"]["persona_seed"])
personas.to_csv(ROOT / cfg["personas"]["file"], index=False)
print(f"{len(personas)} agents, weights sum to {personas.weight.sum():.3f}")
display(personas.groupby("group").agg(n=("agent_id", "size"), weight=("weight", "sum"), can_wfh=("can_wfh", "mean")).round(2))
print("\nExample prompt profile:\n" + persona_text(personas.iloc[0]))

## 6. Provenance และตารางแหล่งที่มาของพารามิเตอร์ (แนวทางหัวข้อ 9.2)

In [ ]:
from datetime import date
provenance = {
    "contact_matrices": {"name": "Prem et al. 2021 synthetic contact matrices (Thailand, THA)",
                         "source": "https://github.com/kieshaprem/synthetic-contact-matrices (output/syntheticmatrices)",
                         "citation": "Prem K, et al. Projecting contact matrices in 177 geographical regions. PLoS Comput Biol 2021;17(7):e1009098",
                         "files": [f"contact_{l}.rdata" for l in ["home", "work", "school", "others"]]},
    "population": {"name": "UN World Population Prospects 2020 by 5-year age (copy in Prem repo)", "source": SOURCES["popage_total2020.csv"]},
    "mobility": {"name": "Google COVID-19 Community Mobility Reports, Thailand national", "source": SOURCES["Region_Mobility_Report_CSVs.zip"],
                 "note": "Archived dataset (reports ended Oct 2022). Used only for plausible ranges."},
    "policy_cases": {"name": "Oxford COVID-19 Government Response Tracker, national compact v1, THA", "source": SOURCES["OxCGRT_compact_national_v1.csv"],
                     "citation": "Hale T, et al. Nat Hum Behav 2021;5:529-538", "note": "Stringency for policy bounds; ConfirmedCases for context only."},
    "persona_marginals": {"name": "Occupation shares, WFH ability, household attributes", "source": "manual",
                          "note": "config/persona_marginals.csv; status column says data or assumption. Fill from NSO Labour Force Survey / Census 2020."},
    "disease": {"name": "SEIR parameters", "source": "Aoki & Ghaffarzadegan (2026); see 04_อธิบายตัวแปรและค่าสถิติ.md"},
    "prepared": str(date.today()),
}
json.dump(provenance, open(ROOT / "data" / "PROVENANCE.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

c, d, pc = cfg["contacts"], cfg["disease"], cfg["policy"]
P = lambda var, val, src, status: {"variable": var, "value": val, "source": src, "status": status}
params = [
    P("beta0", d["beta0"], "Aoki & Ghaffarzadegan 2026", "literature"),
    P("latent_days", d["latent_days"], "Aoki & Ghaffarzadegan 2026", "literature"),
    P("infectious_days", d["infectious_days"], "Aoki & Ghaffarzadegan 2026", "literature"),
    P("symptomatic_frac", d["symptomatic_frac"], "", "assumption"),
    P("N, initial fractions", cfg["population"]["N"], "synthetic population", "assumption"),
    *[P(f"contacts_{r.location}_adult", round(r.adult_20_64, 3), "Prem et al. 2021, THA", "data") for r in contacts.itertuples()],
    P("employment_rate", c["employment_rate"], "replace with NSO LFS employment-to-population ratio 20-64", "assumption"),
    *[P(k, c[k], "", "assumption") for k in ["f_work_other", "f_errand", "f_visit", "f_visit_home"]],
    P("baseline_mix", json.dumps(c["baseline_mix"]), "", "assumption"),
    P("age_groups_20_64", json.dumps(age_groups), "UN WPP 2020", "data"),
    P("household_size", json.dumps(cfg["personas"]["household_size"]), "replace with Census 2020", "assumption"),
    P("p_high_risk_member", cfg["personas"]["p_high_risk_member"], "", "assumption"),
    P("risk_perception, trust_health_info", "see base.yaml", "", "assumption"),
    *[P(f"occupation_share_{r.group}", r.share, r.source_note, r.status) for r in marg.itertuples()],
    P("policy_k", pc["k"], "Aoki & Ghaffarzadegan 2026", "literature"),
    P("policy_alpha", pc["alpha"], "Aoki & Ghaffarzadegan 2026", "literature"),
    P("policy_levels_d", json.dumps({k: v["d"] for k, v in pc["levels"].items()}), "bounded by OxCGRT THA stringency range", "assumption"),
]
ps = pd.DataFrame(params)
assert ps["status"].isin(["data", "literature", "assumption"]).all(), "every parameter needs a status"
ps.to_csv(ROOT / "data" / "parameter_sources.csv", index=False)
display(ps)
print(ps["status"].value_counts().to_dict())